# Page leaf — L1 > L2 > L3 + specialty (Colab first)

One encoder (**BioBERT** by default) with a **leaf** head — one class per whole path `L1 > L2 > L3` of taxonomy v29.3 — and a specialty head. Codeability is not learned: it comes from the leaf's taxonomy tag. **Colab:** Runtime → Change runtime type → **GPU** (T4 or better), then edit `BASE_PATH` and `RUN` only, and Runtime → Run all.

Put these in one Drive folder (`BASE_PATH`):

* `train_leaf.py`, `predict_leaf.py`, `taxonomy.json` (v29.3), `requirements.txt`, `check_test_pages.py`, `test_pages.jsonl` — from `training/bert-training-leaf/`
* `training_data.jsonl` — from `Training/processed/` (3,054 usable pages, 0 unknown labels)

Each run is written to `BASE_PATH/runs/<RUN>/` after every improving epoch, so a Colab disconnect loses at most one epoch. On a PC the same notebook runs from `training/bert-training-leaf/` with `data/` and `output/` there. Page text is never printed — only counts, class names and metrics.

Small classes: whole charts are split train / val / test spreading every leaf and specialty; a leaf with fewer than 3 pages folds into its parent's `Others` (`--min-leaf-pages`); rare leaves repeat up to 4× until they reach 30 pages (`--oversample`); loss weights are √(inverse frequency).

In [ ]:
import json, os, sys, subprocess
from pathlib import Path

ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/page-leaf-training')  # <- edit: the Drive folder
    CODE, DATA, RUNS_DIR = BASE_PATH, BASE_PATH / 'training_data.jsonl', BASE_PATH / 'runs'
else:
    CODE = Path.cwd()                                               # run from training/bert-training-leaf
    DATA, RUNS_DIR = CODE / 'data' / 'training_data.jsonl', CODE / 'output' / 'runs'

# Encoders to compare: same data, seed and split for all. Run the notebook once per RUN
# (change it, then Runtime -> Run all); the last cell compares every finished run.
RUNS = {
    'biobert':         {'base_model': 'dmis-lab/biobert-base-cased-v1.2',  'max_length': 512, 'epochs': 20, 'lr': 3e-5, 'batch_size': 16},
    'bioclinicalbert': {'base_model': 'emilyalsentzer/Bio_ClinicalBERT',   'max_length': 512, 'epochs': 20, 'lr': 3e-5, 'batch_size': 16},
}
RUN = 'biobert'   # <- edit: the run to train now
SETTINGS = RUNS[RUN]
BASE_MODEL = SETTINGS['base_model']   # or a folder holding the weights where Hugging Face is blocked
OUTPUT = RUNS_DIR / RUN

for name in ('train_leaf.py', 'predict_leaf.py', 'taxonomy.json', 'requirements.txt', 'check_test_pages.py', 'test_pages.jsonl'):
    assert (CODE / name).is_file(), f'{name} is missing from {CODE}'
assert DATA.is_file(), f'no training data at {DATA}'
assert json.loads((CODE / 'taxonomy.json').read_text())['version'] == 'v29.3', 'taxonomy.json is not v29.3'
sys.path.insert(0, str(CODE))
print('code:', CODE, '| data:', DATA, '| run:', RUN, SETTINGS, '| output:', OUTPUT)

In [ ]:
if ON_COLAB:
    %pip install -q -r {str(CODE / 'requirements.txt')}

Which device will train: a CUDA GPU when PyTorch sees one, otherwise the CPU (slow: expect tens of minutes per epoch).

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
def train(*args):
    """Run train_leaf.py with this RUN's settings; its output streams below."""
    cmd = [sys.executable, str(CODE / 'train_leaf.py'), '--data', str(DATA), '--output', str(OUTPUT),
           '--taxonomy', str(CODE / 'taxonomy.json'), '--base-model', BASE_MODEL,
           '--max-length', str(SETTINGS['max_length']), '--lr', str(SETTINGS['lr']),
           '--batch-size', str(SETTINGS['batch_size']), *args]
    # Read the child's output line by line: Colab does not reliably show a child process's stdout.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, cwd=CODE) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
    if proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

## Check

Label space (pages left out and why, leaf classes, folds, specialties), the split, two training steps and a minutes-for-the-run estimate. Saves nothing. If it fails with CUDA out of memory, set `batch_size` to 8 in `RUNS`.

In [ ]:
train('--check')

## Train

Prints the running loss every 10 steps, then per epoch: validation leaf macro F1, accuracy, top-5 recall (is the true leaf among the 5 the pipeline keeps) and L1 accuracy. The best epoch by leaf macro F1 is saved to `OUTPUT`; at the end it is reloaded and scored on the held-out **test** charts.

In [ ]:
train('--epochs', str(SETTINGS['epochs']))

## Results on the test charts

In [ ]:
import pandas as pd
summary = json.loads((OUTPUT / 'metrics_summary.json').read_text())
print(json.dumps(summary, indent=2))

Per leaf: training pages, test support, correct, predicted, F1. Weakest first among leaves with test pages.

In [ ]:
per_leaf = pd.read_csv(OUTPUT / 'per_leaf.csv')
pd.set_option('display.max_rows', 200, 'display.max_colwidth', 80)
per_leaf[per_leaf.support > 0].sort_values(['f1', 'support'])

Leaves that trained but had no test pages (cannot be scored here), and the per-L1 view.

In [ ]:
print('no test pages:', int((per_leaf.support == 0).sum()), 'of', len(per_leaf), 'leaf classes')
per_leaf['L1'] = per_leaf.leaf.str.split(' > ').str[0]
by_l1 = per_leaf.groupby('L1').agg(leaves=('leaf', 'count'), train_pages=('train_pages', 'sum'),
                                   support=('support', 'sum'), correct=('correct', 'sum'))
by_l1['accuracy'] = (by_l1.correct / by_l1.support).round(3)
by_l1.sort_values('accuracy')

Folded leaves: too few pages, trained as their parent's `Others`. The pipeline's keyword canon still finds them.

In [ ]:
space = json.loads((OUTPUT / 'label_space.json').read_text())
folded = space.get('folded') or {}
print(len(folded), 'folded;', len(space['leaf']), 'leaf classes;', len(space.get('specialty') or []), 'specialties')
pd.DataFrame(sorted(folded.items()), columns=['leaf', 'trained as'])

## Synthetic test pages

Twelve made-up pages (`test_pages.jsonl`) with the expected leaf: the top leaf PASS/FAIL, whether the expected leaf is among the 5 kept, and the specialty where one is expected. The cell fails if a Lab Report page comes back as Progress Note.

In [ ]:
def check_pages(folder):
    return subprocess.run([sys.executable, str(CODE / 'check_test_pages.py'), '--model', str(folder),
                           '--pages', str(CODE / 'test_pages.jsonl')], cwd=CODE, capture_output=True, text=True)

result = check_pages(OUTPUT)
print(result.stdout, result.stderr)
assert result.returncode == 0, 'a Lab Report test page came back as Progress Note'

## Inference check

Paste a page's text: the kept leaves (best 5, at most 3 per L1) with probability and codeability, and the specialty — what the pipeline receives.

In [ ]:
from train_leaf import decode, encode_texts, load_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer, space = load_model(OUTPUT, device)

def show(text):
    ids = encode_texts(tokenizer, [text], int(space['max_length']), float(space['head_fraction']))[0]
    with torch.no_grad():
        logits = model(torch.tensor([ids], device=device), torch.ones((1, len(ids)), dtype=torch.long, device=device))
    row = decode(logits, space)[0]
    for leaf in row['leaves']:
        print(f"  {leaf['score']:.4f}  {leaf['leaf']}  ({leaf['codeability'] or '-'})")
    if row.get('specialty'):
        print(f"  specialty {row['specialty']} ({row['specialty_confidence']:.2f})")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

box = widgets.Textarea(placeholder='Paste page text here', layout=widgets.Layout(width='100%', height='220px'))
button = widgets.Button(description='Classify', button_style='primary')
result = widgets.Output()

def on_click(_):
    with result:
        result.clear_output()
        show(box.value)

button.on_click(on_click)
display(box, button, result)

In [ ]:
# No widgets? Paste the text between the triple quotes and run this cell.
TEXT = """
paste page text here
"""
show(TEXT)

## Compare the runs

Every finished run under `runs/`, scored on the same test charts. Pick on leaf macro F1, top-5 recall and the synthetic pages together (the keyword canon chooses among the top 5), not on accuracy alone.

In [ ]:
rows = []
for folder in sorted(p for p in RUNS_DIR.iterdir() if (p / 'metrics_summary.json').is_file()):
    s = json.loads((folder / 'metrics_summary.json').read_text())
    sp = json.loads((folder / 'label_space.json').read_text())
    rows.append({'run': folder.name, 'base model': sp.get('base_model'), 'best epoch': s.get('best_epoch'),
                 'test pages': s.get('pages'), 'leaf macro F1': s.get('leaf_macro_f1'),
                 'leaf accuracy': s.get('leaf_accuracy'), 'top-5 recall': s.get('top5_recall'),
                 'L1 accuracy': s.get('l1_accuracy'),
                 'test pages': next((l for l in check_pages(folder).stdout.splitlines() if l.startswith('Top leaf')), '')})
pd.DataFrame(rows)

## Take it to the pipeline

Zips the run (`encoder/`, `heads.pt`, `label_space.json`, metrics) next to it on Drive. Download it, unzip to `core-pipeline/models/page-family/page-leaf/` on each host (weights are not in git), restart, and check `/health` → `page_leaf.ready`.

In [ ]:
import shutil
archive = shutil.make_archive(str(RUNS_DIR / f'page-leaf-{RUN}'), 'zip', root_dir=OUTPUT)
print('wrote', archive, f'({Path(archive).stat().st_size / 1e6:.0f} MB)')